# Notebook 02: Expansion Metrics
## SaaS Expansion Analytics — RavenStack Dataset
**Objective:** Calculate core SaaS metrics — MRR/ARR growth, Net Revenue Retention (NRR), upgrade rates, and expansion MRR opportunity.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded")

Libraries loaded


In [2]:
accounts      = pd.read_csv('../data/raw/ravenstack_accounts.csv')
subscriptions = pd.read_csv('../data/raw/ravenstack_subscriptions.csv')

# Parse dates
accounts['signup_date']     = pd.to_datetime(accounts['signup_date'])
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date']   = pd.to_datetime(subscriptions['end_date'])

# Exclude $0 MRR (trials)
subscriptions = subscriptions[subscriptions['mrr_amount'] > 0]

print(f"Subscriptions loaded: {len(subscriptions):,}")
print(f"After excluding $0 MRR: {len(subscriptions):,}")
print(f"Date range: {subscriptions['start_date'].min().date()} → "
      f"{subscriptions['start_date'].max().date()}")

Subscriptions loaded: 4,222
After excluding $0 MRR: 4,222
Date range: 2023-01-09 → 2024-12-31


## 1. MRR & ARR Overview
Total recurring revenue and growth trajectory.

In [3]:
# Current MRR — active subscriptions only
active_subs = subscriptions[subscriptions['end_date'].isna()]

total_mrr = active_subs['mrr_amount'].sum()
total_arr = total_mrr * 12

print("=== Current Revenue Snapshot ===")
print(f"Active subscriptions: {len(active_subs):,}")
print(f"Total MRR:            ${total_mrr:,.0f}")
print(f"Total ARR:            ${total_arr:,.0f}")

print(f"\nMRR by plan tier:")
mrr_by_plan = (active_subs.groupby('plan_tier')['mrr_amount']
               .agg(['sum', 'mean', 'count'])
               .round(0)
               .sort_values('sum', ascending=False))
mrr_by_plan.columns = ['total_mrr', 'avg_mrr', 'count']
mrr_by_plan['pct_of_mrr'] = (
    mrr_by_plan['total_mrr'] / total_mrr * 100
).round(1)
print(mrr_by_plan)

=== Current Revenue Snapshot ===
Active subscriptions: 3,814
Total MRR:            $10,159,608
Total ARR:            $121,915,296

MRR by plan tier:
            total_mrr  avg_mrr  count  pct_of_mrr
plan_tier                                        
Enterprise    7546876  5787.00   1304       74.30
Pro           1924818  1501.00   1282       18.90
Basic          687914   560.00   1228        6.80


## 2. MRR Growth Over Time
Monthly MRR trend from new subscriptions.

In [4]:
# Monthly MRR from start_date
subscriptions['month'] = subscriptions['start_date'].dt.to_period('M')

monthly_mrr = (subscriptions
               .groupby('month')['mrr_amount']
               .sum()
               .reset_index())
monthly_mrr['month_str'] = monthly_mrr['month'].astype(str)
monthly_mrr['cumulative_mrr'] = monthly_mrr['mrr_amount'].cumsum()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Monthly New MRR', 'Cumulative MRR']
)

fig.add_trace(go.Bar(
    x=monthly_mrr['month_str'],
    y=monthly_mrr['mrr_amount'],
    marker_color='#3498db',
    name='Monthly MRR'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=monthly_mrr['month_str'],
    y=monthly_mrr['cumulative_mrr'],
    mode='lines+markers',
    line=dict(color='#2ecc71', width=2),
    name='Cumulative MRR'
), row=1, col=2)

fig.update_layout(
    height=400,
    font=dict(size=13),
    showlegend=False
)
fig.show()

## 3. Net Revenue Retention (NRR)
NRR measures revenue retained and expanded from existing customers.
NRR > 100% means expansion exceeds churn — the SaaS growth engine.

In [5]:
# NRR = (Starting MRR + Expansion MRR - Churned MRR) / Starting MRR
# Using upgrade_flag as expansion proxy
# Using churn_flag as churned MRR proxy

starting_mrr  = subscriptions['mrr_amount'].sum()
expansion_mrr = (subscriptions[subscriptions['upgrade_flag'] == True]
                 ['mrr_amount'].sum())
churned_mrr   = (subscriptions[subscriptions['churn_flag'] == True]
                 ['mrr_amount'].sum())
contraction_mrr = (subscriptions[subscriptions['downgrade_flag'] == True]
                   ['mrr_amount'].sum())

ending_mrr = starting_mrr + expansion_mrr - churned_mrr - contraction_mrr
nrr        = ending_mrr / starting_mrr * 100

print("=== Net Revenue Retention ===")
print(f"Starting MRR:     ${starting_mrr:>12,.0f}")
print(f"+ Expansion MRR:  ${expansion_mrr:>12,.0f}  "
      f"(upgrades)")
print(f"- Churned MRR:    ${churned_mrr:>12,.0f}  "
      f"(churned subscriptions)")
print(f"- Contraction:    ${contraction_mrr:>12,.0f}  "
      f"(downgrades)")
print(f"{'─'*40}")
print(f"Ending MRR:       ${ending_mrr:>12,.0f}")
print(f"\nNRR:              {nrr:.1f}%")
print(f"\nBenchmark:")
print(f"  > 120% → Best in class (Slack, Snowflake)")
print(f"  > 100% → Healthy growth engine")
print(f"  < 100% → Revenue leaking faster than expanding")

=== Net Revenue Retention ===
Starting MRR:     $  11,338,747
+ Expansion MRR:  $   1,262,997  (upgrades)
- Churned MRR:    $   1,179,139  (churned subscriptions)
- Contraction:    $     459,366  (downgrades)
────────────────────────────────────────
Ending MRR:       $  10,963,239

NRR:              96.7%

Benchmark:
  > 120% → Best in class (Slack, Snowflake)
  > 100% → Healthy growth engine
  < 100% → Revenue leaking faster than expanding


## 4. Upgrade Rate Analysis
Which plans, industries and referral sources produce the most upgrades?

In [6]:
# Active subscriptions only for upgrade analysis
subs_enriched = subscriptions[
    subscriptions['end_date'].isna()
].merge(
    accounts[['account_id', 'industry', 'referral_source', 'country']],
    on='account_id',
    how='left'
)

# Merge subscriptions with accounts for industry/referral context
subs_enriched = subscriptions.merge(
    accounts[['account_id', 'industry', 'referral_source', 'country']],
    on='account_id',
    how='left'
)

# Upgrade rate by plan tier
print("=== Upgrade Rate by Plan Tier ===")
upgrade_by_plan = (subs_enriched.groupby('plan_tier')
                   .agg(
                       total=('subscription_id', 'count'),
                       upgrades=('upgrade_flag', 'sum')
                   )
                   .reset_index())
upgrade_by_plan['upgrade_rate_pct'] = (
    upgrade_by_plan['upgrades'] /
    upgrade_by_plan['total'] * 100
).round(1)
print(upgrade_by_plan.sort_values('upgrade_rate_pct', ascending=False)
      .to_string(index=False))

# Upgrade rate by industry
print("\n=== Upgrade Rate by Industry ===")
upgrade_by_industry = (subs_enriched.groupby('industry')
                       .agg(
                           total=('subscription_id', 'count'),
                           upgrades=('upgrade_flag', 'sum')
                       )
                       .reset_index())
upgrade_by_industry['upgrade_rate_pct'] = (
    upgrade_by_industry['upgrades'] /
    upgrade_by_industry['total'] * 100
).round(1)
print(upgrade_by_industry.sort_values(
    'upgrade_rate_pct', ascending=False
).to_string(index=False))

# Upgrade rate by referral source
print("\n=== Upgrade Rate by Referral Source ===")
upgrade_by_referral = (subs_enriched.groupby('referral_source')
                       .agg(
                           total=('subscription_id', 'count'),
                           upgrades=('upgrade_flag', 'sum')
                       )
                       .reset_index())
upgrade_by_referral['upgrade_rate_pct'] = (
    upgrade_by_referral['upgrades'] /
    upgrade_by_referral['total'] * 100
).round(1)
print(upgrade_by_referral.sort_values(
    'upgrade_rate_pct', ascending=False
).to_string(index=False))

=== Upgrade Rate by Plan Tier ===
 plan_tier  total  upgrades  upgrade_rate_pct
       Pro   1416       168             11.90
Enterprise   1450       160             11.00
     Basic   1356       123              9.10

=== Upgrade Rate by Industry ===
     industry  total  upgrades  upgrade_rate_pct
   HealthTech    786       100             12.70
Cybersecurity    866        99             11.40
     DevTools    984       104             10.60
      FinTech    930        96             10.30
       EdTech    656        52              7.90

=== Upgrade Rate by Referral Source ===
referral_source  total  upgrades  upgrade_rate_pct
        partner    748        90             12.00
          event    777        84             10.80
            ads    838        89             10.60
        organic   1001       105             10.50
          other    858        83              9.70


In [7]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Upgrade Rate by Industry', 
                    'Upgrade Rate by Referral Source']
)

# Industry
ind_sorted = upgrade_by_industry.sort_values('upgrade_rate_pct')
fig.add_trace(go.Bar(
    x=ind_sorted['upgrade_rate_pct'],
    y=ind_sorted['industry'],
    orientation='h',
    text=[f"{x:.1f}%" for x in ind_sorted['upgrade_rate_pct']],
    textposition='outside',
    marker_color='#3498db',
    showlegend=False
), row=1, col=1)

# Referral
ref_sorted = upgrade_by_referral.sort_values('upgrade_rate_pct')
fig.add_trace(go.Bar(
    x=ref_sorted['upgrade_rate_pct'],
    y=ref_sorted['referral_source'],
    orientation='h',
    text=[f"{x:.1f}%" for x in ref_sorted['upgrade_rate_pct']],
    textposition='outside',
    marker_color='#e67e22',
    showlegend=False
), row=1, col=2)

fig.update_layout(
    height=400,
    font=dict(size=13),
    title='Upgrade Rates by Segment'
)
max_val = max(
    upgrade_by_industry['upgrade_rate_pct'].max(),
    upgrade_by_referral['upgrade_rate_pct'].max()
)

fig.update_xaxes(range=[0, max_val * 1.25], row=1, col=1)
fig.update_xaxes(range=[0, max_val * 1.25], row=1, col=2)
fig.show()

## 5. Expansion MRR Opportunity
If all active Basic accounts upgrade to Pro, and all active Pro accounts upgrade to Enterprise — what is the total expansion opportunity?

In [8]:
# Load expansion universe
expansion = pd.read_csv('../data/processed/expansion_universe.csv')

avg_mrr_by_plan = (active_subs.groupby('plan_tier')['mrr_amount']
                   .mean().round(0))

basic_accounts    = expansion[expansion['plan_tier'] == 'Basic']
pro_accounts      = expansion[expansion['plan_tier'] == 'Pro']

avg_basic_mrr      = avg_mrr_by_plan['Basic']
avg_pro_mrr        = avg_mrr_by_plan['Pro']
avg_enterprise_mrr = avg_mrr_by_plan['Enterprise']

basic_opportunity = len(basic_accounts) * (avg_pro_mrr - avg_basic_mrr)
pro_opportunity   = len(pro_accounts) * (avg_enterprise_mrr - avg_pro_mrr)
total_opportunity = basic_opportunity + pro_opportunity

print("=== Expansion MRR Opportunity ===")
print(f"\nBasic → Pro ({len(basic_accounts)} accounts):")
print(f"  Avg MRR uplift:    ${avg_pro_mrr - avg_basic_mrr:,.0f}/mo")
print(f"  Total opportunity: ${basic_opportunity:,.0f}/mo")

print(f"\nPro → Enterprise ({len(pro_accounts)} accounts):")
print(f"  Avg MRR uplift:    ${avg_enterprise_mrr - avg_pro_mrr:,.0f}/mo")
print(f"  Total opportunity: ${pro_opportunity:,.0f}/mo")

print(f"\n{'─'*45}")
print(f"Total expansion opportunity: ${total_opportunity:,.0f}/mo")
print(f"                             ${total_opportunity*12:,.0f}/yr (ARR)")
print(f"\nNote: Assumes 100% conversion — actual realisation")
print(f"      typically 10-20% in a given quarter")
print(f"Realistic quarterly target: "
      f"${total_opportunity * 0.15:,.0f}/mo "
      f"(15% conversion)")

=== Expansion MRR Opportunity ===

Basic → Pro (131 accounts):
  Avg MRR uplift:    $941/mo
  Total opportunity: $123,271/mo

Pro → Enterprise (139 accounts):
  Avg MRR uplift:    $4,286/mo
  Total opportunity: $595,754/mo

─────────────────────────────────────────────
Total expansion opportunity: $719,025/mo
                             $8,628,300/yr (ARR)

Note: Assumes 100% conversion — actual realisation
      typically 10-20% in a given quarter
Realistic quarterly target: $107,854/mo (15% conversion)


## 6. Key Findings

- **NRR is 96.7%** — below the 100% healthy threshold.
  Contraction MRR ($459K from downgrades) is offsetting
  expansion gains. Reducing downgrades is as important
  as driving upgrades.

- **Enterprise generates 74% of MRR** from 34% of subscriptions.
  Protecting Enterprise accounts is the highest revenue priority.

- **Pro → Enterprise is the primary upsell motion.**
  $595K/mo opportunity vs $123K/mo for Basic → Pro.
  Pro accounts should be the CSM team's first focus.

- **HealthTech upgrades at 12.7%** — highest of any industry.
  Partner channel produces the highest upgrade rate (12%).
  Both signal where to focus acquisition and expansion efforts.

- **Total addressable expansion: $8.6M ARR**
  At 15% quarterly conversion: $107K/mo incremental MRR.